# 面试问题：MCP 中 Sampling、Elicitation 和 Roots 能力怎样安全协作？

可以直接复述的回答是：第一，Server 只能使用 Client 在初始化时声明的能力。第二，Roots 是客户端允许访问的资源边界，不是普通字符串前缀。第三，涉及敏感数据或外部动作前应通过 Elicitation 获取明确用户选择。第四，Sampling 请求要限制模型偏好、token 预算和上下文来源。第五，路径规范化、用户拒绝和会话版本必须进入状态机。第六，用能力轨迹和拒绝原因验证实现。下面构建一个离线合规报告助手。

## 真实案例：合规 Server 读取报告、征求同意并请求摘要

客户端公开两个只读 roots：reports 与 policies，并声明 sampling、elicitation、roots 三项能力。六个操作覆盖合法读取、敏感摘要前征询、用户同意、采样请求、越界读取和路径穿越。文件内容使用内存字典，不访问真实磁盘。

In [1]:
from pathlib import PurePosixPath  # 使用纯路径对象规范化教学 URI 而不触碰文件系统
client = {"protocol": "2025-11-25", "capabilities": {"sampling", "elicitation", "roots"}, "roots": ["/workspace/reports", "/workspace/policies"]}  # 定义客户端协商能力与只读根目录
resources = {"/workspace/reports/q2-risk.md": "Q2 风险：供应商延迟率 8%，需管理层复核。", "/workspace/policies/vendor.md": "供应商风险报告必须在发送前获得负责人确认。"}  # 定义两个脱敏内存资源
operations = [  # 定义六个能力与安全边界操作
    {"id": "OP-1", "type": "read", "path": "/workspace/policies/vendor.md"},  # 合法读取政策 root
    {"id": "OP-2", "type": "read", "path": "/workspace/reports/q2-risk.md"},  # 合法读取敏感风险报告
    {"id": "OP-3", "type": "elicit", "ticket": "CONSENT-1", "question": "是否允许使用 Q2 风险报告生成内部摘要？"},  # 请求用户明确同意
    {"id": "OP-4", "type": "sample", "ticket": "CONSENT-1", "prompt": "仅依据 Q2 风险报告生成三点摘要", "max_tokens": 96},  # 同意后请求客户端模型采样
    {"id": "OP-5", "type": "read", "path": "/workspace/secrets.env"},  # 访问未授权 root 的越界请求
    {"id": "OP-6", "type": "read", "path": "/workspace/reports/../secrets.env"},  # 使用字符串前缀伪装的路径穿越
]  # 结束六个协议操作
print("客户端能力：", sorted(client["capabilities"]), "roots=", client["roots"])  # 展示 Server 可使用的协商边界
print("操作输入：id | type | target")  # 展示状态机将处理的真实消息
for operation in operations:  # 逐条输出六个操作
    print(f"{operation['id']} | {operation['type']} | {operation.get('path', operation.get('ticket'))}")  # 呈现合法、需同意和越界请求


客户端能力： ['elicitation', 'roots', 'sampling'] roots= ['/workspace/reports', '/workspace/policies']
操作输入：id | type | target
OP-1 | read | /workspace/policies/vendor.md
OP-2 | read | /workspace/reports/q2-risk.md
OP-3 | elicit | CONSENT-1
OP-4 | sample | CONSENT-1
OP-5 | read | /workspace/secrets.env
OP-6 | read | /workspace/reports/../secrets.env


## Baseline / 基线：字符串 startswith 与默认同意

天真实现只检查路径字符串是否以 root 开头，并假定 Sampling 已获同意。`reports/../secrets.env` 因前缀正确而被错误接受。

In [2]:
def naive_root_allowed(path):  # 实现不做路径规范化的脆弱 root 检查
    return any(path.startswith(root) for root in client["roots"])  # 仅比较原始字符串前缀
traversal_path = operations[-1]["path"]  # 取出包含上级目录的穿越请求
naive_traversal_allowed = naive_root_allowed(traversal_path)  # 使用字符串基线检查恶意路径
naive_sampling_allowed = "sampling" in client["capabilities"]  # 错误地把能力存在等同于用户同意
print("路径穿越：", traversal_path)  # 展示会逃出 reports root 的原始路径
print("字符串前缀是否允许：", naive_traversal_allowed)  # 展示错误 root 边界
print("未征询用户时是否允许敏感 Sampling：", naive_sampling_allowed)  # 展示能力与同意混淆


路径穿越： /workspace/reports/../secrets.env
字符串前缀是否允许： True
未征询用户时是否允许敏感 Sampling： True


## 核心实现：能力、规范 Root 与 Elicitation 状态机

处理器先检查能力，再规范化路径并验证它位于某个 root 内。Sampling 若引用敏感报告，必须携带已同意 ticket，且 token 预算不超过 128。

In [3]:
def normalize_path(path):  # 在内存中解析点段并生成规范绝对路径
    parts = []  # 保存消解后的安全路径组件
    for part in PurePosixPath(path).parts:  # 逐级处理根、普通组件和上级目录
        if part in {"/", ".", ""}:  # 根标记和当前目录不进入组件栈
            continue  # 跳过不承载资源身份的部分
        if part == "..":  # 上级目录需要弹出前一组件
            if parts:  # 仅在存在父级时向上移动
                parts.pop()  # 消解一个目录层级
            continue  # 完成点段处理
        parts.append(part)  # 保存普通目录或文件名
    return "/" + "/".join(parts)  # 返回规范化绝对路径
def within_root(path, root):  # 按路径组件而非字符串检查 root 边界
    normalized_path = PurePosixPath(normalize_path(path))  # 规范化请求路径
    normalized_root = PurePosixPath(normalize_path(root))  # 规范化协商 root
    return normalized_path == normalized_root or normalized_root in normalized_path.parents  # 仅允许 root 本身或真实后代
class CapabilitySession:  # 实现 Sampling、Elicitation 与 Roots 的最小状态机
    def __init__(self, client_contract):  # 保存初始化时协商的客户端合同
        self.capabilities = set(client_contract["capabilities"])  # 复制能力防止外部修改
        self.roots = list(client_contract["roots"])  # 复制只读 roots
        self.consents = {}  # 保存用户对 elicitation ticket 的决定
        self.trace = []  # 保存操作、状态和拒绝原因
    def handle(self, operation):  # 处理单个能力操作并记录结果
        operation_type = operation["type"]  # 读取 read、elicit 或 sample 类型
        if operation_type == "read":  # 资源读取必须使用 roots 能力
            normalized = normalize_path(operation["path"])  # 先消解路径穿越点段
            allowed = "roots" in self.capabilities and any(within_root(normalized, root) for root in self.roots)  # 检查能力和规范父子关系
            result = {"status": "ok", "content": resources.get(normalized, "not_found"), "path": normalized} if allowed else {"status": "denied", "reason": "outside_roots", "path": normalized}  # 返回内容或明确边界错误
        elif operation_type == "elicit":  # 用户征询需要客户端支持 elicitation
            result = {"status": "pending", "ticket": operation["ticket"], "question": operation["question"]} if "elicitation" in self.capabilities else {"status": "denied", "reason": "elicitation_not_supported"}  # 创建待用户决定票据
        elif operation_type == "sample":  # Server 请求客户端模型生成摘要
            ticket = operation.get("ticket")  # 读取敏感采样绑定的同意票据
            if "sampling" not in self.capabilities:  # 未协商 sampling 时不能发起模型调用
                result = {"status": "denied", "reason": "sampling_not_supported"}  # 返回能力缺失
            elif self.consents.get(ticket) is not True:  # 用户未明确同意或已拒绝时禁止采样
                result = {"status": "denied", "reason": "consent_required"}  # 返回同意门禁错误
            elif operation["max_tokens"] > 128:  # 限制 Server 可请求的生成预算
                result = {"status": "denied", "reason": "token_budget_exceeded"}  # 防止无限采样成本
            else:  # 能力、同意和预算均通过
                result = {"status": "ok", "sample": "- 延迟率为 8%\n- 需要管理层复核\n- 仅限内部使用", "max_tokens": operation["max_tokens"]}  # 返回确定性教学摘要
        else:  # 未知能力操作不能静默执行
            result = {"status": "denied", "reason": "unknown_operation"}  # 返回协议合同错误
        self.trace.append((operation["id"], operation_type, result["status"], result.get("reason", "-")))  # 写入可审计能力轨迹
        return result  # 返回当前操作结果
    def resolve(self, ticket, approved):  # 接收用户对 Elicitation 的明确选择
        self.consents[ticket] = bool(approved)  # 保存同意或拒绝而非只记录票据存在
session = CapabilitySession(client)  # 创建协商完成的教学会话
policy_result = session.handle(operations[0])  # 读取供应商政策
report_result = session.handle(operations[1])  # 读取 Q2 风险报告
elicitation_result = session.handle(operations[2])  # 创建敏感摘要同意票据
sample_before_consent = session.handle(operations[3])  # 在用户回应前尝试采样
session.resolve("CONSENT-1", True)  # 模拟用户明确选择允许内部摘要
sample_after_consent = session.handle(operations[3])  # 使用同一票据重新请求采样
print("能力轨迹：operation | type | status | reason")  # 输出能力、同意和状态迁移
for event in session.trace:  # 逐条展示五次处理结果
    print(" | ".join(event))  # 让 consent_required 到成功的变化可见
print("同意后的摘要：", sample_after_consent.get("sample"))  # 展示通过门禁后的真实可读结果


能力轨迹：operation | type | status | reason
OP-1 | read | ok | -
OP-2 | read | ok | -
OP-3 | elicit | pending | -
OP-4 | sample | denied | consent_required
OP-4 | sample | ok | -
同意后的摘要： - 延迟率为 8%
- 需要管理层复核
- 仅限内部使用


## 失败案例与修正：路径穿越与 root 外读取

规范化后 `reports/../secrets.env` 变成 `/workspace/secrets.env`，不再属于 reports root。直接越界和伪装穿越都返回 outside_roots，且不会读取宿主文件。

In [4]:
outside_result = session.handle(operations[4])  # 处理直接访问未授权 secrets.env 的请求
traversal_result = session.handle(operations[5])  # 处理带前缀伪装的路径穿越请求
oversized_sample = dict(operations[3])  # 复制合法采样请求构造预算反例
oversized_sample["id"] = "OP-7"  # 为错误请求分配独立操作身份
oversized_sample["max_tokens"] = 4096  # 请求远超协商上限的生成预算
oversized_result = session.handle(oversized_sample)  # 应用 token 预算门禁
print("直接越界：", outside_result)  # 展示规范路径和拒绝原因
print("路径穿越：", traversal_result)  # 展示点段消解后的真实目标
print("超预算 Sampling：", oversized_result)  # 展示同意存在也不能绕过资源预算


直接越界： {'status': 'denied', 'reason': 'outside_roots', 'path': '/workspace/secrets.env'}
路径穿越： {'status': 'denied', 'reason': 'outside_roots', 'path': '/workspace/secrets.env'}
超预算 Sampling： {'status': 'denied', 'reason': 'token_budget_exceeded'}


## 结果表：能力操作的允许与拒绝

In [5]:
result_rows = [("read_policy", policy_result), ("read_report", report_result), ("sample_before_consent", sample_before_consent), ("sample_after_consent", sample_after_consent), ("outside_root", outside_result), ("path_traversal", traversal_result), ("oversized_sampling", oversized_result)]  # 汇总七个关键能力结果
print("case | status | reason_or_path")  # 输出同一状态口径的能力对照表
for name, result in result_rows:  # 逐案例展示允许和拒绝
    detail = result.get("reason", result.get("path", "authorized"))  # 选择最能解释结果的字段
    print(f"{name:22} | {result['status']:7} | {detail}")  # 展示同意、root 和预算门禁各自作用
allowed_count = sum(result["status"] == "ok" for _, result in result_rows)  # 统计真正执行的安全操作数
denied_count = sum(result["status"] == "denied" for _, result in result_rows)  # 统计被门禁拒绝的操作数
print(f"汇总：allowed={allowed_count}，denied={denied_count}，consents={session.consents}")  # 输出会话安全状态


case | status | reason_or_path
read_policy            | ok      | /workspace/policies/vendor.md
read_report            | ok      | /workspace/reports/q2-risk.md
sample_before_consent  | denied  | consent_required
sample_after_consent   | ok      | authorized
outside_root           | denied  | outside_roots
path_traversal         | denied  | outside_roots
oversized_sampling     | denied  | token_budget_exceeded
汇总：allowed=3，denied=4，consents={'CONSENT-1': True}


## 结果解读

客户端声明 sampling 并不等于用户同意读取敏感报告：OP-4 在 consent 前失败，明确批准后才成功。字符串前缀接受的穿越路径经规范化后落到 roots 外，被可靠拒绝。Token 预算是与同意独立的资源门禁，防止已授权请求无限消耗客户端模型。

## 生产边界

完整 MCP 实现需依据实际规范版本处理 capability schema、取消、传输断连和用户输入验证。文件 URI 还涉及符号链接、大小写、挂载点和 TOCTOU，必须依赖操作系统级 sandbox，不能只用字符串逻辑。Elicitation 不应索取密码或秘密，Sampling 上下文需最小化并可审计。

## 最小回归测试

In [6]:
assert len(operations) >= 5  # 保证案例覆盖多种能力与边界操作
assert naive_traversal_allowed is True  # 保证字符串前缀基线真实暴露路径穿越
assert sample_before_consent["reason"] == "consent_required"  # 保证能力存在不能绕过用户同意
assert sample_after_consent["status"] == "ok"  # 保证明确同意后受限 Sampling 可以完成
assert outside_result["reason"] == "outside_roots" and traversal_result["reason"] == "outside_roots"  # 保证直接和伪装越界均被拒绝
assert oversized_result["reason"] == "token_budget_exceeded"  # 保证已同意请求仍受生成预算约束
